# M1 — Case type  (summary-only, group-aware split)

Based on **Louis**'s case_type model: TF-IDF of the summary into logistic
regression. Strengthened with character n-grams, class weighting, and a
**group-aware split** (identical summaries never straddle train/test).
Trains, evaluates on the held-out group, refits on all data, and saves
`models/m1_case_type.joblib`.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import northstar_prep as P
from sklearn.metrics import f1_score, accuracy_score

df = P.load_clean()
FEATS, TARGET = ["case_summary"], "case_type"
tr, te = P.group_split(df)                         # dedupe-safe split
pipe = P.make_clf([P.t_word(), P.t_char()]).fit(tr[FEATS], tr[TARGET])
pred = pipe.predict(te[FEATS])
metrics = {"macro_F1": round(f1_score(te[TARGET], pred, average="macro"), 3),
           "accuracy": round(accuracy_score(te[TARGET], pred), 3),
           "baseline_acc": round(df[TARGET].value_counts(normalize=True).iloc[0], 3)}
print(metrics)

In [ ]:
# held-out table the app will display (predictions from the split model)
proba = pipe.predict_proba(te[FEATS])
test = te[["case_id","case_summary",TARGET]].copy().reset_index(drop=True)
test["predicted"]  = pred
test["confidence"] = proba.max(axis=1).round(3)

full = P.make_clf([P.t_word(), P.t_char()]).fit(df[FEATS], df[TARGET])   # deploy model
path = P.save_bundle("m1_case_type", {
    "model": full, "target": TARGET, "features": FEATS, "classes": sorted(df[TARGET].unique()),
    "kind": "multiclass", "label": "Case type", "metrics": metrics, "test": test,
    "meta": {"author":"Louis-style","split":"group-on-summary","inputs":"summary only"}})
print("saved", path, "| test rows:", len(test))